# 손동작 *모션* 분류 — GRU 시퀀스 모델 → ONNX (Colab)

1. 노트북에서 녹화: `live_record.py` (웹캠 / `--source realsense`) → `~/data/assist_arm/gestures/<label>/*.npz`
2. 업로드 zip 만들기: `python3 ~/cobot2_ws/src/cobot2/assist_arm/tools/db_export_colab.py` (DB 기준, 라벨·카메라 골라서)
   → `~/data/assist_arm/export/assist_arm_<시각>.zip` (gestures/ + features.py + gesture_dataset.py + 이 노트북 + db/*.csv)
   → zip 을 drive.google.com **내 드라이브**에 올리고, Colab 새 셀에서 `!unzip -q /content/drive/MyDrive/assist_arm_<시각>.zip -d /content/drive/MyDrive/` → `MyDrive/assist_arm`
3. Drive 의 `assist_arm/colab_gesture_seq_gru.ipynb` 를 Colab 으로 열고 [런타임 → 모두 실행] (GPU 없어도 됨, 수 분) → `gesture_seq.onnx`, `gesture_seq_labels.json`
4. Drive 의 `assist_arm/gesture_seq.onnx`, `gesture_seq_labels.json` 을 내려받아 노트북 `~/data/assist_arm/models/` 에 넣기
   (gesture_node 기본 경로 그대로라 설정 수정 불필요, gesture_node 재시작)

입력 (1, 30, 71): 프레임마다 손 기준 좌표 63 (손목 원점·손바닥 축으로 회전 정규화, world landmarks 우선) + 손바닥 방향 3 + 손바닥 좌표계 속도 3 + 속력 1 + 손 있음 1 
웹캠·RealSense 녹화를 섞어 학습 가능 (카메라가 달라도 비슷한 특징). 증강: 좌우 반전, 3D 회전 ±40°, 스케일, 노이즈.

In [ ]:
from google.colab import drive; drive.mount('/content/drive')
BASE = '/content/drive/MyDrive/assist_arm'
import sys, os, json; sys.path.insert(0, BASE)
!pip -q install onnx onnxruntime
import numpy as np, torch, torch.nn as nn
from gesture_dataset import load_samples, sequence_dataset
from features import SEQ_LEN, SEQ_FEATURE_DIM, FEATURE_VERSION
samples = load_samples(f'{BASE}/gestures')
# 손 기준 좌표 + 손바닥 좌표계 속도 (features v2), 좌우 반전 + 3D 회전·스케일·노이즈 증강 5배
X, y, who, srcs = sequence_dataset(samples, SEQ_LEN, mirror=True, augment=5)
labels = sorted(set(y)); labels = ['idle'] + [l for l in labels if l != 'idle'] if 'idle' in labels else labels
Y = np.array([labels.index(v) for v in y])
print('cameras', dict(zip(*np.unique(srcs, return_counts=True))), X.shape, {l: int((Y == i).sum()) for i, l in enumerate(labels)}, 'persons', sorted(set(who)))

## 증강 + 사람 단위 검증 분할

In [ ]:
rng = np.random.default_rng(0)
def augment(x):                                 # 3D 회전·스케일·노이즈는 sequence_dataset 에서 이미 함 → 여기선 시작 시점만 흔들기
    return np.roll(x, rng.integers(-4, 5), axis=1)

persons = sorted(set(who))
val_person = persons[-1] if len(persons) > 1 else None
va = (who == val_person) if val_person else (rng.random(len(X)) < 0.2)
Xtr, Ytr, Xva, Yva = X[~va], Y[~va], X[va], Y[va]
print('val person:', val_person, 'train', len(Xtr), 'val', len(Xva))

## GRU 모델

In [ ]:
class GestureGRU(nn.Module):
    def __init__(self, f=SEQ_FEATURE_DIM, h=64, n=len(labels)):
        super().__init__()
        self.gru = nn.GRU(f, h, num_layers=2, batch_first=True, dropout=0.2)
        self.head = nn.Sequential(nn.Linear(h, 64), nn.ReLU(), nn.Dropout(0.2), nn.Linear(64, n))
    def forward(self, x):                      # x: (B, T, F)
        o, _ = self.gru(x)
        return self.head(o[:, -1])

dev = 'cuda' if torch.cuda.is_available() else 'cpu'
model = GestureGRU().to(dev)
opt = torch.optim.AdamW(model.parameters(), 2e-3, weight_decay=1e-4)
lossf = nn.CrossEntropyLoss()
best, best_state = 0, None
for ep in range(80):
    model.train()
    xa = torch.from_numpy(augment(Xtr)).to(dev); ya = torch.from_numpy(Ytr).to(dev)
    perm = torch.randperm(len(xa))
    for i in range(0, len(xa), 64):
        idx = perm[i:i + 64]; opt.zero_grad(); lossf(model(xa[idx]), ya[idx]).backward(); opt.step()
    model.eval()
    with torch.no_grad():
        pv = model(torch.from_numpy(Xva).to(dev)).argmax(1).cpu().numpy()
    acc = (pv == Yva).mean() if len(Yva) else 0
    if acc >= best: best, best_state = acc, {k: v.clone() for k, v in model.state_dict().items()}
    if (ep + 1) % 10 == 0: print(f'epoch {ep + 1} val acc {acc:.3f}')
model.load_state_dict(best_state); print('best val acc', best)

## 혼동 행렬

In [ ]:
model.eval()
with torch.no_grad():
    pv = model(torch.from_numpy(Xva).to(dev)).argmax(1).cpu().numpy()
cm = np.zeros((len(labels), len(labels)), int)
for t, p in zip(Yva, pv): cm[t, p] += 1
print(labels); print(cm)

## ONNX 내보내기 (softmax 포함, 입력 이름 'seq')

In [ ]:
export = nn.Sequential(model.cpu(), nn.Softmax(dim=1)).eval()
torch.onnx.export(export, torch.zeros(1, SEQ_LEN, SEQ_FEATURE_DIM), f'{BASE}/gesture_seq.onnx',
                  input_names=['seq'], output_names=['prob'], dynamic_axes={'seq': {0: 'n'}, 'prob': {0: 'n'}},
                  opset_version=17)
json.dump({'labels': labels, 'feature_version': FEATURE_VERSION}, open(f'{BASE}/gesture_seq_labels.json', 'w'))
import onnxruntime as ort
s = ort.InferenceSession(f'{BASE}/gesture_seq.onnx')
print(s.get_inputs()[0].shape, s.run(None, {'seq': Xva[:1]})[0].round(3), labels)